In [2]:
def validate_transition_matrix(P, tolerance=1e-9):
    n = len(P)
    if n == 0:
        raise ValueError("Transition matrix cannot be empty.")

    for row in P:
        if len(row) != n:
            raise ValueError("Transition matrix must be square.")
    for i in range(n):
        row_sum = sum(P[i])
        if abs(row_sum - 1) > tolerance:
            raise ValueError(f"Row {i} of the transition matrix does not sum to 1 (sum={row_sum}).")
        
        for j in range(n):
            if P[i][j] < 0:
                raise ValueError(f"Transition probability P[{i}][{j}] is negative (value={P[i][j]}).")


def build_graph(P, tolerance=1e-12):
    n = len(P)
    graph = {i: [] for i in range(n)}
    print(graph)
    for i in range(n):
        for j in range(n):
            if P[i][j] > tolerance:
                graph[i].append(j)
    return graph

In [12]:
def validate_transition_matrix(P, tolerance=1e-9):
    """
    Checks whether P is a valid transition matrix.

    Conditions:
    1. P must be square.
    2. All probabilities must be non-negative.
    3. Every row must sum to 1.
    """

    n = len(P)

    if n == 0:
        raise ValueError("The transition matrix cannot be empty.")

    for row in P:
        if len(row) != n:
            raise ValueError("The transition matrix must be square.")

    for i in range(n):
        row_sum = sum(P[i])

        if abs(row_sum - 1.0) > tolerance:
            raise ValueError(f"Row {i} does not sum to 1. Sum = {row_sum}")

        for j in range(n):
            if P[i][j] < -tolerance:
                raise ValueError(f"Negative probability found at P[{i}][{j}].")


def build_graph(P, tolerance=1e-12):
    """
    Creates a directed graph from the transition matrix.

    There is an edge i -> j if P[i][j] > 0.
    """

    n = len(P)
    graph = {i: [] for i in range(n)}

    for i in range(n):
        for j in range(n):
            if P[i][j] > tolerance:
                graph[i].append(j)

    return graph


def tarjan_scc(graph):
    """
    Tarjan's algorithm for finding strongly connected components.

    Each strongly connected component corresponds to one communicating class.

    This version prints the full progress of the algorithm in order to help
    understand how indices, lowlink values, the stack, and components evolve.
    """

    print("\n==============================")
    print("TARJAN'S ALGORITHM STARTED")
    print("==============================\n")

    print("Input graph:")
    for state, neighbors in graph.items():
        print(f"State {state} -> {neighbors}")

    print("\nExplanation:")
    print("Each key is a state.")
    print("Each list contains the states that can be reached directly from that state.")
    print("The algorithm will now find the strongly connected components.")
    print("In Markov Chains, these are the communicating classes.\n")

    index = 0
    stack = []
    on_stack = set()

    indices = {}
    lowlink = {}

    components = []

    def print_current_status():
        """
        Helper function to print the current internal state of the algorithm.
        """

        print("Current algorithm status:")
        print(f"  indices = {indices}")
        print(f"  lowlink = {lowlink}")
        print(f"  stack   = {stack}")
        print(f"  on_stack = {on_stack}")
        print()

    def strongconnect(v, depth=0):
        """
        Recursive DFS function used by Tarjan's algorithm.

        The depth variable is only used to indent the printed output,
        so that the recursion is easier to read.
        """

        nonlocal index

        indent = "    " * depth

        print(f"{indent}--------------------------------")
        print(f"{indent}Visit state {v}")
        print(f"{indent}--------------------------------")

        indices[v] = index
        lowlink[v] = index

        print(f"{indent}Assign:")
        print(f"{indent}  indices[{v}] = {indices[v]}")
        print(f"{indent}  lowlink[{v}] = {lowlink[v]}")

        index += 1

        stack.append(v)
        on_stack.add(v)

        print(f"{indent}Push state {v} onto the stack.")
        print(f"{indent}Stack is now: {stack}")
        print(f"{indent}on_stack is now: {on_stack}")
        print()

        print(f"{indent}Explore neighbors of state {v}: {graph[v]}")

        for w in graph[v]:
            print()
            print(f"{indent}Checking edge {v} -> {w}")

            if w not in indices:
                print(f"{indent}State {w} has not been visited yet.")
                print(f"{indent}Call strongconnect({w}).")

                strongconnect(w, depth + 1)

                old_lowlink = lowlink[v]
                lowlink[v] = min(lowlink[v], lowlink[w])

                print(f"{indent}Returned to state {v} from state {w}.")
                print(f"{indent}Update lowlink[{v}] using lowlink[{w}].")
                print(f"{indent}  old lowlink[{v}] = {old_lowlink}")
                print(f"{indent}  lowlink[{w}] = {lowlink[w]}")
                print(f"{indent}  new lowlink[{v}] = {lowlink[v]}")

            elif w in on_stack:
                print(f"{indent}State {w} has already been visited and is still on the stack.")
                print(f"{indent}This means edge {v} -> {w} connects back to an active state.")

                old_lowlink = lowlink[v]
                lowlink[v] = min(lowlink[v], indices[w])

                print(f"{indent}Update lowlink[{v}] using indices[{w}].")
                print(f"{indent}  old lowlink[{v}] = {old_lowlink}")
                print(f"{indent}  indices[{w}] = {indices[w]}")
                print(f"{indent}  new lowlink[{v}] = {lowlink[v]}")

            else:
                print(f"{indent}State {w} has already been assigned to a completed component.")
                print(f"{indent}No lowlink update is needed.")

        print()
        print(f"{indent}Finished exploring all neighbors of state {v}.")
        print(f"{indent}Check whether state {v} is the root of a component.")
        print(f"{indent}  lowlink[{v}] = {lowlink[v]}")
        print(f"{indent}  indices[{v}] = {indices[v]}")

        if lowlink[v] == indices[v]:
            print(f"{indent}Because lowlink[{v}] == indices[{v}],")
            print(f"{indent}state {v} is the root of a strongly connected component.")

            component = []

            print(f"{indent}Start popping states from the stack until state {v} is removed.")

            while True:
                w = stack.pop()
                on_stack.remove(w)
                component.append(w)

                print(f"{indent}  Pop state {w}")
                print(f"{indent}  Current component: {component}")
                print(f"{indent}  Stack after pop: {stack}")

                if w == v:
                    break

            components.append(component)

            print(f"{indent}Completed strongly connected component: {component}")
            print(f"{indent}In Markov Chain terms, this is one communicating class.")
        else:
            print(f"{indent}Because lowlink[{v}] != indices[{v}],")
            print(f"{indent}state {v} is not the root of a component yet.")

        print()
        print(f"{indent}Status after finishing state {v}:")
        print(f"{indent}  indices = {indices}")
        print(f"{indent}  lowlink = {lowlink}")
        print(f"{indent}  stack   = {stack}")
        print(f"{indent}  on_stack = {on_stack}")
        print()

    print("Begin DFS traversal.\n")

    for v in graph:
        if v not in indices:
            print("================================================")
            print(f"State {v} has not been visited yet.")
            print(f"Start a new DFS search from state {v}.")
            print("================================================\n")

            strongconnect(v)

        else:
            print(f"State {v} has already been visited, so we skip it.")

    print("\n==============================")
    print("TARJAN'S ALGORITHM FINISHED")
    print("==============================\n")

    print("Final strongly connected components:")
    for class_id, component in enumerate(components, start=1):
        print(f"Component {class_id}: {component}")

    print("\nFinal indices:")
    print(indices)

    print("\nFinal lowlink values:")
    print(lowlink)

    print("\nFinal stack:")
    print(stack)

    print("\nInterpretation:")
    print("Each component is a group of states where every state can reach every other state.")
    print("For a Markov Chain, each component is a communicating class.")
    print("After this step, we check whether each class is closed.")
    print("Closed communicating classes are recurrent.")
    print("Non-closed communicating classes are transient.\n")

    return components


def is_closed_class(component, graph):
    """
    Checks whether a communicating class is closed.

    A class is closed if no state inside the class can move
    to a state outside the class.

    Returns:
        True  -> closed class -> recurrent states
        False -> non-closed class -> transient states
    """

    print("\n" + "=" * 65)
    print(f"CHECKING WHETHER COMPONENT {component} IS CLOSED")
    print("=" * 65)

    # Convert list to set for faster membership checking.
    component_set = set(component)

    print(f"Component as a set: {component_set}")
    print("\nWe examine every outgoing edge from every state in this class.")
    print("If even one edge goes outside the class, the class is NOT closed.\n")

    # Check every state that belongs to the communicating class.
    for state in component:
        print(f"Current state: {state}")
        print(f"Possible transitions from state {state}: {graph[state]}")

        # Check every possible destination of the current state.
        for next_state in graph[state]:
            print(f"  Checking transition: {state} -> {next_state}")

            # If destination is not inside the component, the class can be escaped.
            if next_state not in component_set:
                print(f"  State {next_state} is NOT inside the component {component_set}.")
                print(f"  Therefore, the transition {state} -> {next_state} leaves the class.")
                print("\nRESULT: This component is NOT CLOSED.")
                print("Therefore, all states in this component are TRANSIENT.")

                return False

            print(f"  State {next_state} is inside the component.")

        print(f"All transitions from state {state} remain inside the component.\n")

    # If the loop finishes, no outgoing edge leaves the component.
    print("No transition from this component leads outside it.")
    print("RESULT: This component IS CLOSED.")
    print("Therefore, all states in this component are RECURRENT.")

    return True



def analyze_markov_chain(P):
    """
    Complete Markov Chain analysis.

    Steps:
    1. Validate the transition matrix.
    2. Build the directed graph.
    3. Use Tarjan's algorithm to find communicating classes.
    4. Check whether every communicating class is closed.
    5. Classify states as recurrent or transient.

    Returns:
        result     : dictionary with the classification of every state
        components : communicating classes found by Tarjan's algorithm
    """

    print("\n" + "#" * 75)
    print("MARKOV CHAIN ANALYSIS STARTED")
    print("#" * 75)

    # ----------------------------------------------------------
    # Step 1: Validate the transition matrix
    # ----------------------------------------------------------
    print("\nSTEP 1: VALIDATE THE TRANSITION MATRIX")
    print("-" * 75)

    validate_transition_matrix(P)

    print("The transition matrix is valid.")
    print("Every row sums to 1 and all probabilities are non-negative.")

    # ----------------------------------------------------------
    # Step 2: Build the directed graph
    # ----------------------------------------------------------
    print("\nSTEP 2: BUILD THE DIRECTED GRAPH")
    print("-" * 75)

    graph = build_graph(P)

    print("Graph created from the transition matrix:")
    print("An edge i -> j exists when P[i][j] > 0.\n")

    for state, neighbors in graph.items():
        print(f"State {state} -> {neighbors}")

    # ----------------------------------------------------------
    # Step 3: Find communicating classes using Tarjan
    # ----------------------------------------------------------
    print("\nSTEP 3: FIND COMMUNICATING CLASSES WITH TARJAN'S ALGORITHM")
    print("-" * 75)

    components = tarjan_scc(graph)

    print("\nTarjan's algorithm returned the following components:")

    for component_number, component in enumerate(components, start=1):
        print(f"Communicating class {component_number}: {component}")

    print("\nImportant:")
    print("At this point, we know which states communicate.")
    print("We still need to determine whether each class is closed.")
    print("Only closed communicating classes are recurrent.")

    # ----------------------------------------------------------
    # Step 4: Prepare result dictionary
    # ----------------------------------------------------------
    result = {}

    # This number is only assigned to recurrent classes.
    recurrent_class_id = 1

    # ----------------------------------------------------------
    # Step 5: Check every communicating class
    # ----------------------------------------------------------
    print("\nSTEP 4: CHECK WHETHER EACH COMMUNICATING CLASS IS CLOSED")
    print("-" * 75)

    for component_number, component in enumerate(components, start=1):
        print("\n" + "*" * 75)
        print(f"ANALYZING COMMUNICATING CLASS {component_number}: {component}")
        print("*" * 75)

        closed = is_closed_class(component, graph)

        # ------------------------------------------------------
        # Case A: Closed component -> recurrent states
        # ------------------------------------------------------
        if closed:
            print("\nThis is a closed communicating class.")
            print(f"It will receive recurrent class ID = {recurrent_class_id}")

            for state in component:
                result[state] = {
                    "type": "recurrent",
                    "recurrent_class": recurrent_class_id
                }

                print(
                    f"State {state} is classified as RECURRENT "
                    f"in recurrent class {recurrent_class_id}."
                )

            recurrent_class_id += 1

        # ------------------------------------------------------
        # Case B: Non-closed component -> transient states
        # ------------------------------------------------------
        else:
            print("\nThis is a non-closed communicating class.")
            print("All states in this class are transient.")

            for state in component:
                result[state] = {
                    "type": "transient",
                    "recurrent_class": None
                }

                print(
                    f"State {state} is classified as TRANSIENT "
                    f"because its class can be escaped."
                )

    # ----------------------------------------------------------
    # Step 6: Print final result
    # ----------------------------------------------------------
    print("\n" + "#" * 75)
    print("FINAL CLASSIFICATION OF STATES")
    print("#" * 75)

    for state in sorted(result):
        state_type = result[state]["type"]
        class_id = result[state]["recurrent_class"]

        if state_type == "recurrent":
            print(
                f"State {state}: RECURRENT "
                f"(recurrent class ID = {class_id})"
            )
        else:
            print(f"State {state}: TRANSIENT")

    print("\nInterpretation:")
    print("- Recurrent states belong to closed communicating classes.")
    print("- Transient states belong to classes with an exit to another class.")
    print("- A recurrent class ID is shared by all recurrent states in the same class.")

    print("\n" + "#" * 75)
    print("MARKOV CHAIN ANALYSIS FINISHED")
    print("#" * 75)

    return result, components


def print_analysis(result):
    """
    Prints the final classification of each state.
    """

    for state in sorted(result):
        state_type = result[state]["type"]
        recurrent_class = result[state]["recurrent_class"]

        if state_type == "recurrent":
            print(f"State {state}: recurrent, recurrent class {recurrent_class}")
        else:
            print(f"State {state}: transient")

In [13]:
P = [
    [1.0, 0.0, 0.0, 0.0],
    [0.2, 0.3, 0.4, 0.1],
    [0.0, 0.2, 0.0, 0.8],
    [0.0, 0.0, 0.0, 1.0]
]

result, components = analyze_markov_chain(P)

print("Communicating classes:")
print(components)

print("\nState classification:")
print_analysis(result)


###########################################################################
MARKOV CHAIN ANALYSIS STARTED
###########################################################################

STEP 1: VALIDATE THE TRANSITION MATRIX
---------------------------------------------------------------------------
The transition matrix is valid.
Every row sums to 1 and all probabilities are non-negative.

STEP 2: BUILD THE DIRECTED GRAPH
---------------------------------------------------------------------------
Graph created from the transition matrix:
An edge i -> j exists when P[i][j] > 0.

State 0 -> [0]
State 1 -> [0, 1, 2, 3]
State 2 -> [1, 3]
State 3 -> [3]

STEP 3: FIND COMMUNICATING CLASSES WITH TARJAN'S ALGORITHM
---------------------------------------------------------------------------

TARJAN'S ALGORITHM STARTED

Input graph:
State 0 -> [0]
State 1 -> [0, 1, 2, 3]
State 2 -> [1, 3]
State 3 -> [3]

Explanation:
Each key is a state.
Each list contains the states that can be reached directly 

In [ ]:
#AI solution

import numpy as np

# 1. Define the transition matrix
P = np.array([
    [1.0, 0.0, 0.0, 0.0],
    [0.2, 0.3, 0.4, 0.1],
    [0.0, 0.2, 0.0, 0.8],
    [0.0, 0.0, 0.0, 1.0]
])

print("--- Step 1: Initial Transition Matrix ---")
print(P)

# 2. Determine reachability
# If a state can reach another in n steps, (P^n)_ij > 0
reach = np.linalg.matrix_power(P > 0, 4)
print("\n--- Step 2: Reachability Matrix ---")
print(reach)

# 3. Identify communication classes
n_states = P.shape[0]
communicates = np.zeros((n_states, n_states), dtype=bool)
for i in range(n_states):
    for j in range(n_states):
        communicates[i, j] = reach[i, j] and reach[j, i]
        print(f"reach[{i}, {j}] = {reach[i, j]}, reach[{j}, {i}] = {reach[j, i]}")
        print(f"State {i} communicates with state {j}: {communicates[i, j]}")
        
print("\n--- Step 3: Communication Matrix (i <-> j) ---")
print(communicates.astype(int))

# 4. Group states into classes and check for recurrence
visited = set()
for i in range(n_states):
    if i not in visited:
        class_members = [j for j in range(n_states) if communicates[i, j]]
        print(f"\nClass found: {class_members}")    
        visited.update(class_members)
        
        # Check if the class is closed
        is_closed = True
        for member in class_members:
            for k in range(n_states):
                if k not in class_members and P[member, k] > 0:
                    is_closed = False
                    print(f"Class {class_members} is NOT closed: "
                          f"State {member} can move to state {k} outside.")
                    break
        
        if is_closed:
            print(f"Class {class_members} is CLOSED (Recurrent).")
        else:
            print(f"Class {class_members} is NOT closed (Transient).")

--- Step 1: Initial Transition Matrix ---
[[1.  0.  0.  0. ]
 [0.2 0.3 0.4 0.1]
 [0.  0.2 0.  0.8]
 [0.  0.  0.  1. ]]

--- Step 2: Reachability Matrix ---
[[ True False False False]
 [ True  True  True  True]
 [ True  True  True  True]
 [False False False  True]]
reach[0, 0] = True, reach[0, 0] = True
State 0 communicates with state 0: True
reach[0, 1] = False, reach[1, 0] = True
State 0 communicates with state 1: False
reach[0, 2] = False, reach[2, 0] = True
State 0 communicates with state 2: False
reach[0, 3] = False, reach[3, 0] = False
State 0 communicates with state 3: False
reach[1, 0] = True, reach[0, 1] = False
State 1 communicates with state 0: False
reach[1, 1] = True, reach[1, 1] = True
State 1 communicates with state 1: True
reach[1, 2] = True, reach[2, 1] = True
State 1 communicates with state 2: True
reach[1, 3] = True, reach[3, 1] = False
State 1 communicates with state 3: False
reach[2, 0] = True, reach[0, 2] = False
State 2 communicates with state 0: False
reach[2, 1]

In [8]:
P.shape[0]

4